In [9]:
from typing import Tuple, Union
import math
import struct

Number = Union[int, float, complex, Tuple[float, float]]
ComplexPair = Tuple[float, float]

def _as_complex(value: Number) -> complex:
    if isinstance(value, tuple):
        if len(value) != 2:
            raise ValueError("Complex pairs must have exactly two elements.")
        return complex(float(value[0]), float(value[1]))
    return complex(value)

def _as_float(value: Number) -> float:
    if isinstance(value, tuple):
        if len(value) != 2:
            raise ValueError("Real inputs must have exactly two elements.")
        if abs(float(value[1])) > 1e-12:
            raise ValueError("pade_sqrt accepts real-valued inputs only.")
        return float(value[0])
    if isinstance(value, complex):
        if abs(value.imag) > 1e-12:
            raise ValueError("pade_sqrt accepts real-valued inputs only.")
        return float(value.real)
    return float(value)

def _to_pair(value: complex) -> ComplexPair:
    return (float(value.real), float(value.imag))

def _sqrt_initial_guess(value: float) -> float:
    if value == 0.0:
        return 0.0
    if value < 0.0:
        raise ValueError("pade_sqrt accepts non-negative real inputs only.")

    bits = struct.unpack(">I", struct.pack(">f", float(value)))[0]
    exponent_bits = (bits >> 23) & 0xFF

    if exponent_bits == 0:
        mantissa = float(value)
        exponent = 0
        while mantissa < 1.0:
            mantissa *= 2.0
            exponent -= 1
        guess_exponent = (exponent + 127) // 2
    elif exponent_bits == 0xFF:
        raise ValueError("pade_sqrt does not support NaN or infinity.")
    else:
        guess_exponent = (exponent_bits + 127) // 2

    return math.ldexp(1.0, int(guess_exponent) - 127)

def _scale_exp_approx(exp_bits: int) -> int:
    return ((exp_bits + 2 * 127) * 0b1011) >> 5

def _cbrt_initial_guess(value: complex) -> complex:
    s = _as_complex(value)
    if s == 0:
        return 0j

    re = float(s.real)
    im = float(s.imag)

    # Work with raw 32-bit float bitpatterns
    re_bits = struct.unpack(">I", struct.pack(">f", re))[0]
    im_bits = struct.unpack(">I", struct.pack(">f", im))[0]

    # Extract sign bit (0x80000000) and exponent (bits 23..30)
    re_sign_bit = re_bits & 0x80000000
    im_sign_bit = im_bits & 0x80000000

    e_re = (re_bits >> 23) & 0xFF
    e_im = (im_bits >> 23) & 0xFF

    if e_re == 0xFF or e_im == 0xFF:
        raise ValueError("pade_cbrt does not support NaN or infinity.")

    # Treat subnormals by promoting exponent to 1 when magnitude non-zero
    if e_re == 0 and re != 0.0:
        e_re = 1
    if e_im == 0 and im != 0.0:
        e_im = 1

    # Use the larger exponent to get a conservative seed magnitude
    e_max = max(e_re, e_im)

    # Compute approximate cube-root exponent: floor((E-127)/3)+127
    e_cbrt = ((e_max - 127) // 3) + 127

    # Clamp into valid exponent range for IEEE-754 single precision
    if e_cbrt <= 0:
        e_cbrt = 1
    elif e_cbrt >= 0xFF:
        e_cbrt = 0xFE

    # Build seed bitpatterns with zero mantissa and chosen exponent,
    # preserving sign bits from original inputs.
    re_seed_bits = re_sign_bit | (e_cbrt << 23)
    im_seed_bits = im_sign_bit | (e_cbrt << 23)

    re_seed = struct.unpack(">f", struct.pack(">I", re_seed_bits))[0]
    im_seed = struct.unpack(">f", struct.pack(">I", im_seed_bits))[0]

    # If original component was exactly zero, keep seed exactly zero
    if re == 0.0:
        re_seed = 0.0
    if im == 0.0:
        im_seed = 0.0
    return complex(re_seed, im_seed)

def pade_sqrt(z: Number, iterations: int = 5) -> ComplexPair:
    s = _as_float(z)
    if s == 0.0:
        return (0.0, 0.0)

    p = _sqrt_initial_guess(s)
    for _ in range(max(1, int(iterations))):
        p2 = p * p
        denominator = 3.0 * p2 + s
        if abs(denominator) < 1e-18:
            break
        p = p * (p2 + 3.0 * s) / denominator

    return (float(p), 0.0)

def pade_cbrt(z: Number, iterations: int = 5) -> ComplexPair:
    s = _as_complex(z)
    if s == 0:
        return (0.0, 0.0)

    p = _cbrt_initial_guess(s)
    for _ in range(max(1, int(iterations))):
        p3 = p * p * p
        denominator = 2.0 * p3 + s
        if abs(denominator) < 1e-18:
            break
        p = p * (p3 + 2.0 * s) / denominator

    return _to_pair(p)


In [10]:
import random

# ==========================================
# TEST SECTION (TESTBENCH)
# ==========================================
def run_benchmark(num_tests=10000, sqrt_iterations=10, cbrt_iterations=10):
    random.seed(7)

    # 500 positive real tests for sqrt
    real_samples = [random.uniform(0.0, 1000.0) for _ in range(num_tests // 2)]

    # 500 complex tests for cbrt
    complex_samples = [complex(random.uniform(-500.0, 500.0), random.uniform(-500.0, 500.0)) for _ in range(num_tests // 2)]

    sqrt_errors = []
    cbrt_errors = []
    threshold = 1e-3

    sqrt_pass = 0
    cbrt_pass = 0

    for sample in real_samples:
        my_sqrt = complex(*pade_sqrt(float(sample), iterations=sqrt_iterations))
        rel_error = abs(my_sqrt**2 - sample) / sample if sample != 0 else 0.0
        sqrt_errors.append(rel_error)
        if rel_error < threshold:
            sqrt_pass += 1

    for sample in complex_samples:
        my_cbrt = complex(*pade_cbrt(sample, iterations=cbrt_iterations))
        rel_error = abs(my_cbrt**3 - sample) / (abs(sample) if sample != 0 else 1.0)
        cbrt_errors.append(rel_error)
        if rel_error < threshold:
            cbrt_pass += 1

    print(f"Total sqrt test cases: {len(real_samples)}")
    print(f"Total cbrt test cases: {len(complex_samples)}")
    print(f"\n--- SQRT RESULTS ({sqrt_iterations} iterations) ---")
    print(f"Passed tests (relative error < {threshold}): {sqrt_pass}/{len(real_samples)}")
    print(f"Average error: {sum(sqrt_errors)/len(sqrt_errors):.6f}")
    print(f"Maximum error: {max(sqrt_errors):.6f}")

    print(f"\n--- CBRT RESULTS ({cbrt_iterations} iterations) ---")
    print(f"Passed tests (relative error < {threshold}): {cbrt_pass}/{len(complex_samples)}")
    print(f"Average error: {sum(cbrt_errors)/len(cbrt_errors):.6f}")
    print(f"Maximum error: {max(cbrt_errors):.6f}")

# Quick run
run_benchmark(10000, sqrt_iterations=16, cbrt_iterations=16)


Total sqrt test cases: 5000
Total cbrt test cases: 5000

--- SQRT RESULTS (16 iterations) ---
Passed tests (relative error < 0.001): 5000/5000
Average error: 0.000000
Maximum error: 0.000000

--- CBRT RESULTS (16 iterations) ---
Passed tests (relative error < 0.001): 5000/5000
Average error: 0.000000
Maximum error: 0.000000


In [11]:
EPS = complex(-0.5, 0.8660254037844386)

def solve_cubic(a, b, c, d, sqrt_iterations=5, cbrt_iterations=5):
    b_n = b / (-3.0 * a)
    c_n = c / (-3.0 * a)
    d_n = d / (-3.0 * a)

    delta_0 = b_n**2 + c_n
    delta_1 = 2.0 * b_n**3 + 3.0 * b_n * c_n + 3.0 * d_n
    delta = delta_1**2 - 4.0 * delta_0**3

    roots = []

    if delta.real < 0:
        sqrt_delta_imag = pade_sqrt(abs(float(delta.real)), iterations=sqrt_iterations)[0]
        sqrt_delta = complex(0.0, sqrt_delta_imag)
        C_val = complex(*pade_cbrt((delta_1 + sqrt_delta) / 2.0, iterations=cbrt_iterations))
        C_conj = C_val.conjugate()
        for k in range(3):
            roots.append(b_n + (EPS**k) * C_val + (EPS**(2 * k)) * C_conj)
    elif abs(4.0 * delta_0**3) < 1e-9 and delta_1 <= 0:
        C_val = complex(*pade_cbrt(delta_1, iterations=cbrt_iterations))
        for k in range(3):
            roots.append(b_n + (EPS**k) * C_val)
    else:
        sqrt_delta = complex(*pade_sqrt(float(delta.real), iterations=sqrt_iterations))
        C_val = complex(*pade_cbrt((delta_1 + sqrt_delta) / 2.0, iterations=cbrt_iterations))
        for k in range(3):
            roots.append(b_n + (EPS**k) * C_val + (EPS**(2 * k)) * (delta_0 / C_val))

    return roots

In [12]:
import random
import numpy as np

def sort_roots(roots):
    return sorted([complex(root) for root in roots], key=lambda value: (round(value.real, 4), round(value.imag, 4)))

def _fmt_c(z):
    z = complex(z)
    return f"({z.real:.6f}{'+' if z.imag>=0 else '-'}{abs(z.imag):.6f}j)"

def diagnose_cubic(a, b, c, d, sqrt_iterations=10, cbrt_iterations=10):
    """Compute diagnostic values used by solve_cubic for a given cubic.
    Returns delta_0, delta_1, delta, sqrt_delta, and C_val.
    """
    b_n = b / (-3.0 * a)
    c_n = c / (-3.0 * a)
    d_n = d / (-3.0 * a)

    delta_0 = b_n**2 + c_n
    delta_1 = 2.0 * b_n**3 + 3.0 * b_n * c_n + 3.0 * d_n
    delta = delta_1**2 - 4.0 * delta_0**3

    sqrt_pair = pade_sqrt(abs(float(delta.real)), iterations=sqrt_iterations)
    sqrt_delta = complex(0.0, sqrt_pair[0]) if delta.real < 0 else complex(*sqrt_pair)

    C_val = complex(*pade_cbrt((delta_1 + sqrt_delta) / 2.0, iterations=cbrt_iterations))

    return {
        'b_n': b_n,
        'c_n': c_n,
        'd_n': d_n,
        'delta_0': delta_0,
        'delta_1': delta_1,
        'delta': delta,
        'sqrt_delta': sqrt_delta,
        'C_val': C_val,
    }

def run_cubic_benchmark(num_tests=1000, sqrt_iterations=10, cbrt_iterations=10, report_failures=False, max_report=20):
    random.seed(7)
    pass_count = 0
    threshold = 1e-2
    max_err = 0.0

    failures = []

    for index in range(1, num_tests + 1):
        a = random.uniform(-10.0, 10.0)
        while abs(a) < 1e-3:
            a = random.uniform(-10.0, 10.0)
        b = random.uniform(-10.0, 10.0)
        c = random.uniform(-10.0, 10.0)
        d = random.uniform(-10.0, 10.0)

        my_roots = solve_cubic(a, b, c, d, sqrt_iterations=sqrt_iterations, cbrt_iterations=cbrt_iterations)
        std_roots = np.roots([a, b, c, d])

        my_roots_sorted = sort_roots(my_roots)
        std_roots_sorted = sort_roots(std_roots)

        case_error = 0.0
        for my_root, std_root in zip(my_roots_sorted, std_roots_sorted):
            error = abs(my_root - std_root)
            if error > case_error:
                case_error = error

        if case_error > max_err:
            max_err = case_error

        if case_error < threshold:
            pass_count += 1
        elif report_failures:
            diagnostic = diagnose_cubic(a, b, c, d, sqrt_iterations=sqrt_iterations, cbrt_iterations=cbrt_iterations)
            failures.append({
                'index': index,
                'a': a,
                'b': b,
                'c': c,
                'd': d,
                'err': case_error,
                'my_roots': my_roots_sorted,
                'std_roots': std_roots_sorted,
                'diag': diagnostic,
            })

    print(f"Total test cases: {num_tests}")
    print(f"Matched cases (error < {threshold}): {pass_count}/{num_tests}")
    print(f"Maximum error recorded: {max_err:.6f}")

    if report_failures and failures:
        reported = min(max_report, len(failures))
        print(f"\n--- Failed test cases ({reported}/{len(failures)}) ---")
        header = f"{'#':>4}  {'a':>9}  {'b':>9}  {'c':>9}  {'d':>9}  {'max_err':>10}"
        print(header)
        print('-' * len(header))
        for failure in failures[:max_report]:
            print(f"{failure['index']:4d}  {failure['a']:9.4f}  {failure['b']:9.4f}  {failure['c']:9.4f}  {failure['d']:9.4f}  {failure['err']:10.6e}")

        print('\n--- Root comparison and diagnostics for the first failed cases ---')
        for failure in failures[:min(5, max_report)]:
            diagnostic = failure['diag']
            print(f"\nTest #{failure['index']}: a={failure['a']:.6f}, b={failure['b']:.6f}, c={failure['c']:.6f}, d={failure['d']:.6f}, err={failure['err']:.6e}")
            print(f"  my_roots : {[ _fmt_c(root) for root in failure['my_roots'] ]}")
            print(f"  np.roots : {[ _fmt_c(root) for root in failure['std_roots'] ]}")
            print(f"  delta_0  : {diagnostic['delta_0']}")
            print(f"  delta_1  : {diagnostic['delta_1']}")
            print(f"  delta    : {diagnostic['delta']}")
            print(f"  sqrt_delta: {diagnostic['sqrt_delta']}")
            print(f"  C_val    : {diagnostic['C_val']}")

# Example quick run (smaller number to inspect failures):
run_cubic_benchmark(100000, sqrt_iterations=5, cbrt_iterations=5, report_failures=True, max_report=10)


Total test cases: 100000
Matched cases (error < 0.01): 100000/100000
Maximum error recorded: 0.001810


In [13]:
from pathlib import Path
import random
import struct


def resolve_notebook_path(filename: str) -> Path:
    cwd = Path.cwd()
    if (cwd / "Lab3").is_dir():
        return cwd / "Lab3" / filename
    return cwd / filename


def float_to_fp32_hex(value: float) -> str:
    return f"{struct.unpack('>I', struct.pack('>f', float(value)))[0]:08x}"


def generate_random_tests(output_file: str = "random_tests.txt", num_cases: int = 1000, seed: int = 7, low: float = -10.0, high: float = 10.0, min_abs_a: float = 1e-3):
    random.seed(seed)
    output_path = resolve_notebook_path(output_file)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    with output_path.open("w", encoding="utf-8") as file_handle:
        for _ in range(num_cases):
            a = random.uniform(low, high)
            while abs(a) < min_abs_a:
                a = random.uniform(low, high)
            b = random.uniform(low, high)
            c = random.uniform(low, high)
            d = random.uniform(low, high)
            file_handle.write(
                f"{float_to_fp32_hex(a)} {float_to_fp32_hex(b)} {float_to_fp32_hex(c)} {float_to_fp32_hex(d)}\n"
            )

    print(f"Wrote {num_cases} FP32 hex test cases to: {output_path}")
    return output_path


input_file = generate_random_tests(num_cases=1000, seed=7)


Wrote 1000 FP32 hex test cases to: c:\Users\chith\Downloads\Study\HK252\VLSI\Lab\CE_VLSI_1\Lab3\random_tests.txt


In [ ]:
import struct
import math


def fp32_hex_to_float(token: str) -> float:
    clean = token.strip().lower().replace("0x", "").replace("32'h", "")
    return struct.unpack('>f', struct.pack('>I', int(clean, 16)))[0]


def read_non_empty_lines(path: Path):
    with path.open("r", encoding="utf-8") as file_handle:
        return [line.strip() for line in file_handle if line.strip() and not line.lstrip().startswith("#")]


def parse_input_line(line: str):
    parts = [part for part in line.replace(",", " ").split() if part]
    if len(parts) != 4:
        raise ValueError(f"Each input line must contain 4 FP32 hex coefficients, got {len(parts)}: {line!r}")
    return tuple(fp32_hex_to_float(part) for part in parts)


def parse_output_line(line: str):
    parts = [part for part in line.replace(",", " ").split() if part]
    if len(parts) == 6 and all(len(part.replace("0x", "").replace("32'h", "")) <= 8 for part in parts):
        values = [fp32_hex_to_float(part) for part in parts]
        return [complex(values[index], values[index + 1]) for index in range(0, 6, 2)]
    if len(parts) == 3:
        return [complex(part.strip("()[]{}")) for part in parts]
    if len(parts) == 6:
        values = [float(part) for part in parts]
        return [complex(values[index], values[index + 1]) for index in range(0, 6, 2)]
    if len(parts) == 1:
        return [complex(parts[0].strip("()[]{}"))]
    raise ValueError(f"Could not parse output format: {line!r}")


def _is_finite_complex(value: complex) -> bool:
    value = complex(value)
    return math.isfinite(value.real) and math.isfinite(value.imag)


def check_random_tests(input_file: str = "random_tests.txt", output_file: str = "sim/cubic_solver_output.txt", sqrt_iterations: int = 8, cbrt_iterations: int = 16, max_cases: int = 1000, rel_tol_percent: float = 0.1, abs_tol: float = 1e-6):
    input_path = resolve_notebook_path(input_file)
    output_path = resolve_notebook_path(output_file)

    input_lines = read_non_empty_lines(input_path)
    output_lines = read_non_empty_lines(output_path)

    case_count = min(len(input_lines), len(output_lines), int(max_cases))
    if case_count == 0:
        raise ValueError("No comparable test cases were found.")

    if len(input_lines) != len(output_lines) or case_count != len(input_lines) or case_count != len(output_lines):
        print(f"Warning: using first {case_count} cases only (input={len(input_lines)}, output={len(output_lines)}, max_cases={int(max_cases)}).")

    total_abs_error = 0.0
    total_rel_percent = 0.0
    valid_case_count = 0
    worst_abs_error = 0.0
    worst_rel_error = 0.0
    worst_index = -1
    bad_cases = []
    skipped_cases = []
    nonzero_expected_cases = 0

    for index, (input_line, output_line) in enumerate(zip(input_lines[:case_count], output_lines[:case_count]), start=1):
        a, b, c, d = parse_input_line(input_line)
        golden_roots = sort_roots(solve_cubic(a, b, c, d, sqrt_iterations=sqrt_iterations, cbrt_iterations=cbrt_iterations))
        predicted_roots = sort_roots(parse_output_line(output_line))

        if len(predicted_roots) != 3:
            raise ValueError(f"Output row {index} must contain 3 roots, got {len(predicted_roots)}: {output_line!r}")

        if any(not _is_finite_complex(root) for root in predicted_roots + golden_roots):
            skipped_cases.append((index, input_line, output_line))
            continue

        valid_case_count += 1
        case_abs_error = 0.0
        case_rel_percent = None
        for predicted_root, golden_root in zip(predicted_roots, golden_roots):
            abs_error = abs(predicted_root - golden_root)
            if abs(golden_root) != 0.0:
                rel_percent = (abs_error / abs(golden_root)) * 100.0
            else:
                rel_percent = None
            case_abs_error = max(case_abs_error, abs_error)
            if rel_percent is not None:
                case_rel_percent = rel_percent if case_rel_percent is None else max(case_rel_percent, rel_percent)

        total_abs_error += case_abs_error
        if case_rel_percent is not None:
            total_rel_percent += case_rel_percent
            nonzero_expected_cases += 1

        if case_abs_error > worst_abs_error:
            worst_abs_error = case_abs_error
            worst_rel_error = case_rel_percent if case_rel_percent is not None else worst_rel_error
            worst_index = index

        # Mark as bad: prefer percent-based comparison when expected != 0, otherwise use absolute tolerance
        if case_rel_percent is None:
            if case_abs_error > abs_tol:
                bad_cases.append((index, case_abs_error, None, input_line, output_line))
        else:
            if case_rel_percent > rel_tol_percent:
                bad_cases.append((index, case_abs_error, case_rel_percent, input_line, output_line))

    pass_count = valid_case_count - len(bad_cases)
    fail_count = len(bad_cases)

    print(f"Total test cases checked: {case_count}")
    print(f"Valid comparable cases: {valid_case_count}")
    print(f"Passed cases (rel_err% < {rel_tol_percent}%): {pass_count}")
    print(f"Failed cases: {fail_count}")
    print(f"Skipped non-finite cases: {len(skipped_cases)}")

    if valid_case_count == 0:
        print("No valid cases remained after filtering.")
        return

    # Average relative error as percentage vs expected
    if valid_case_count > 0:
        if nonzero_expected_cases > 0:
            avg_rel_percent = total_rel_percent / nonzero_expected_cases
        else:
            avg_rel_percent = float('nan')
        print(f"Average relative error: {avg_rel_percent:.6f}%")
        print(f"Average absolute error: {total_abs_error / valid_case_count:.6e}")
        print(f"Maximum absolute error: {worst_abs_error:.6e}")
        if worst_rel_error is not None:
            print(f"Maximum relative error: {worst_rel_error:.6f}%")
        else:
            print(f"Maximum relative error: N/A (all expected==0)")
        print(f"Worst test case index: {worst_index}")

    if skipped_cases:
        print("\n--- Skipped non-finite cases ---")
        for index, input_line, output_line in skipped_cases[:10]:
            print(f"#{index}")
            print(f"  input : {input_line}")
            print(f"  output: {output_line}")

    if bad_cases:
        print(f"\n--- Test cases exceeding threshold (rel% > {rel_tol_percent}% or abs > {abs_tol}) ---")
        for index, abs_error, rel_error, input_line, output_line in bad_cases[:10]:
            if rel_error is None:
                print(f"#{index}: abs={abs_error:.6e}, rel=N/A (expected==0)")
            else:
                print(f"#{index}: abs={abs_error:.6e}, rel={rel_error:.6e}%")
            print(f"  input : {input_line}")
            print(f"  output: {output_line}")

    return {"checked": case_count, "passed": pass_count, "failed": fail_count, "skipped": len(skipped_cases)}

# Quick run: invoke checker with defaults
check_random_tests()

Total test cases checked: 1000
Valid comparable cases: 1000
Passed cases (rel_err < 0.001): 987
Failed cases: 13
Skipped non-finite cases: 0
Average relative error: 0.122529%
Average absolute error: 1.868348e-03
Maximum absolute error: 1.802094e+00
Maximum relative error: 111.817274%
Worst test case index: 268

--- Test cases with relative error > {rel_tol} ---
#94: abs=1.931226e-02, rel=1.579090e-02
  input : 3ae8425c c0ce24e2 c043d6b4 c11a3014
  output: 45633ec3 00000000 be735000 3f9c0800 be735000 bf9c0800
#97: abs=4.912798e-03, rel=5.069461e-03
  input : 40cc1be0 bfada019 bdccbc4d 40d62726
  output: bf75a822 00000000 3f160c98 3f6001e3 3f160c98 bf6001e3
#248: abs=9.079027e-03, rel=8.969937e-03
  input : c102a6dc bf94906b 3e501a35 c112f462
  output: bf8dad9f 00000000 3ef6f821 3f666fac 3ef6f821 bf666fac
#268: abs=1.802094e+00, rel=1.118173e+00
  input : 40148bf0 bfad7bdd 3e81d294 40fd25ad
  output: 3eff58e1 00000000 3d2e83d0 be82773c 3d2e83d0 3e82773c
#376: abs=1.025574e-02, rel=2.7266

{'checked': 1000, 'passed': 987, 'failed': 13, 'skipped': 0}

In [15]:
from collections import Counter, defaultdict
from pathlib import Path
import math


def _pairwise_min_distance(values):
    return min(
        abs(values[0] - values[1]),
        abs(values[0] - values[2]),
        abs(values[1] - values[2]),
    )


def _cubic_branch(a, b, c, d):
    b_n = b / (-3.0 * a)
    c_n = c / (-3.0 * a)
    d_n = d / (-3.0 * a)
    delta_0 = b_n**2 + c_n
    delta_1 = 2.0 * b_n**3 + 3.0 * b_n * c_n + 3.0 * d_n
    delta = delta_1**2 - 4.0 * delta_0**3

    if delta.real < 0:
        branch = "negative-discriminant"
    elif abs(4.0 * delta_0**3) < 1e-9 and delta_1 <= 0:
        branch = "near-zero-discriminant"
    else:
        branch = "positive-discriminant"

    return branch, delta_0, delta_1, delta


def _print_table(headers, rows):
    widths = [len(str(header)) for header in headers]
    for row in rows:
        for index, value in enumerate(row):
            widths[index] = max(widths[index], len(str(value)))

    def format_row(row):
        return " | ".join(str(value).ljust(widths[index]) for index, value in enumerate(row))

    separator = "-+-".join("-" * width for width in widths)
    print(format_row(headers))
    print(separator)
    for row in rows:
        print(format_row(row))


def analyze_failed_tests(
    input_file: str = "random_tests.txt",
    output_file: str = "sim/cubic_solver_output.txt",
    sqrt_iterations: int = 5,
    cbrt_iterations: int = 5,
    max_cases: int = 900,
    error_threshold: float = 1e-3,
    near_multiple_threshold: float = 1e-2,
    large_root_threshold: float = 10.0,
):
    input_path = resolve_notebook_path(input_file)
    output_path = resolve_notebook_path(output_file)

    input_lines = read_non_empty_lines(input_path)
    output_lines = read_non_empty_lines(output_path)
    case_count = min(len(input_lines), len(output_lines), int(max_cases))

    counts = Counter()
    branch_fail_counts = Counter()
    branch_error_sums = defaultdict(lambda: [0.0, 0.0, 0])
    overall_error_sum = [0.0, 0.0, 0]
    worst_cases = []
    failed_indices = []

    for index, (input_line, output_line) in enumerate(zip(input_lines[:case_count], output_lines[:case_count]), start=1):
        a, b, c, d = parse_input_line(input_line)
        predicted_roots = sort_roots(parse_output_line(output_line))
        golden_roots = sort_roots(solve_cubic(a, b, c, d, sqrt_iterations=sqrt_iterations, cbrt_iterations=cbrt_iterations))

        finite = all(_is_finite_complex(root) for root in predicted_roots + golden_roots)
        if not finite:
            counts["non-finite"] += 1
            worst_cases.append((index, float("inf"), float("inf"), "non-finite", 0.0, 0.0, input_line, output_line))
            continue

        case_abs_error = 0.0
        case_rel_error = 0.0
        for predicted_root, golden_root in zip(predicted_roots, golden_roots):
            abs_error = abs(predicted_root - golden_root)
            rel_error = abs_error / max(abs(golden_root), 1e-12)
            case_abs_error = max(case_abs_error, abs_error)
            case_rel_error = max(case_rel_error, rel_error)

        if case_abs_error <= error_threshold:
            counts["pass"] += 1
            continue

        branch, delta_0, delta_1, delta = _cubic_branch(a, b, c, d)
        min_root_gap = _pairwise_min_distance(golden_roots)
        max_root_mag = max(abs(root) for root in golden_roots)

        counts["fail"] += 1
        counts[branch] += 1
        branch_fail_counts[branch] += 1
        branch_error_sums[branch][0] += case_abs_error
        branch_error_sums[branch][1] += case_rel_error
        branch_error_sums[branch][2] += 1
        overall_error_sum[0] += case_abs_error
        overall_error_sum[1] += case_rel_error
        overall_error_sum[2] += 1
        failed_indices.append(index)
        if min_root_gap < near_multiple_threshold:
            counts["near-multiple-root"] += 1
        if max_root_mag > large_root_threshold:
            counts["large-root-scale"] += 1

        worst_cases.append(
            (
                index,
                case_abs_error,
                case_rel_error,
                branch,
                min_root_gap,
                max_root_mag,
                input_line,
                output_line,
            )
        )

    worst_cases.sort(key=lambda item: item[1], reverse=True)

    valid_cases = counts["pass"] + counts["fail"]
    print(f"Total checked: {case_count}")
    print(f"Valid comparable cases: {valid_cases}")
    print(f"Passed cases: {counts['pass']}")
    print(f"Failed cases: {counts['fail']}")
    print(f"Non-finite cases: {counts['non-finite']}")
    print(f"Failures in negative-discriminant branch: {counts['negative-discriminant']}")
    print(f"Failures in near-zero-discriminant branch: {counts['near-zero-discriminant']}")
    print(f"Failures in positive-discriminant branch: {counts['positive-discriminant']}")
    print(f"Failures with near-multiple roots (< {near_multiple_threshold:g}): {counts['near-multiple-root']}")
    print(f"Failures with large root scale (> {large_root_threshold:g}): {counts['large-root-scale']}")

    if failed_indices:
        print("\n--- Failed case indices ---")
        _print_table(["case"], [[index] for index in failed_indices])

    if branch_fail_counts:
        print("\n--- Average error by failure branch ---")
        branch_rows = []
        for name, count in branch_fail_counts.most_common():
            abs_sum, rel_sum, branch_count = branch_error_sums[name]
            branch_rows.append([
                name,
                branch_count,
                f"{abs_sum / branch_count:.6e}",
                f"{rel_sum / branch_count:.6e}",
            ])
        _print_table(["branch", "count", "mean_abs", "mean_rel"], branch_rows)

    if overall_error_sum[2] > 0:
        print("\n--- Overall failure averages ---")
        _print_table(
            ["metric", "value"],
            [
                ["mean_abs", f"{overall_error_sum[0] / overall_error_sum[2]:.6e}"],
                ["mean_rel", f"{overall_error_sum[1] / overall_error_sum[2]:.6e}"],
            ],
        )

    print("\n--- Worst failures ---")
    worst_rows = []
    for item in worst_cases[:10]:
        index, abs_error, rel_error, branch, min_root_gap, max_root_mag, input_line, output_line = item
        worst_rows.append([
            index,
            f"{abs_error:.6e}",
            f"{rel_error:.6e}",
            branch,
            f"{min_root_gap:.6e}",
            f"{max_root_mag:.6e}",
        ])
    _print_table(["case", "abs", "rel", "branch", "min_gap", "max|root|"], worst_rows)

    return {
        "checked": case_count,
        "passed": counts["pass"],
        "failed": counts["fail"],
        "non_finite": counts["non-finite"],
        "by_branch": dict(branch_fail_counts),
        "near_multiple": counts["near-multiple-root"],
        "large_root_scale": counts["large-root-scale"],
        "failed_indices": failed_indices,
    }


analysis_summary = analyze_failed_tests()
print("\nSummary:")
print(analysis_summary)

Total checked: 900
Valid comparable cases: 900
Passed cases: 893
Failed cases: 7
Non-finite cases: 0
Failures in negative-discriminant branch: 0
Failures in near-zero-discriminant branch: 0
Failures in positive-discriminant branch: 7
Failures with near-multiple roots (< 0.01): 0
Failures with large root scale (> 10): 1

--- Failed case indices ---
case
----
94  
97  
248 
268 
376 
448 
650 

--- Average error by failure branch ---
branch                | count | mean_abs     | mean_rel    
----------------------+-------+--------------+-------------
positive-discriminant | 7     | 2.642386e-01 | 1.687259e-01

--- Overall failure averages ---
metric   | value       
---------+-------------
mean_abs | 2.642386e-01
mean_rel | 1.687259e-01

--- Worst failures ---
case | abs          | rel          | branch                | min_gap      | max|root|   
-----+--------------+--------------+-----------------------+--------------+-------------
268  | 1.802094e+00 | 1.118173e+00 | positive-discri

In [16]:
from collections import defaultdict, Counter


def parse_debug_line(line: str):
    parts = [part for part in line.replace(",", " ").split() if part]
    if len(parts) != 5:
        raise ValueError(f"Debug line must contain exactly 5 tokens, got {len(parts)}: {line!r}")
    delta = fp32_hex_to_float(parts[0])
    delta_0 = fp32_hex_to_float(parts[1])
    delta_1 = fp32_hex_to_float(parts[2])
    c_re = fp32_hex_to_float(parts[3])
    c_im = fp32_hex_to_float(parts[4])
    return delta, delta_0, delta_1, complex(c_re, c_im)


def analyze_failed_tests_with_debug(
    input_file: str = "random_tests.txt",
    debug_file: str = "sim/cubic_solver_debug.txt",
    sqrt_iterations: int = 5,
    cbrt_iterations: int = 5,
    max_cases: int = 900,
    error_threshold: float = 1e-3,
):
    if "failed_indices" not in analysis_summary:
        local_summary = analyze_failed_tests(
            input_file=input_file,
            sqrt_iterations=sqrt_iterations,
            cbrt_iterations=cbrt_iterations,
            max_cases=max_cases,
            error_threshold=error_threshold,
        )
        failed_indices = set(local_summary.get("failed_indices", []))
    else:
        failed_indices = set(analysis_summary.get("failed_indices", []))

    debug_path = resolve_notebook_path(debug_file)
    input_path = resolve_notebook_path(input_file)

    if not debug_path.exists():
        print(f"Debug file not found: {debug_path}")
        return None

    debug_lines = read_non_empty_lines(debug_path)
    input_lines = read_non_empty_lines(input_path)
    case_count = min(len(debug_lines), len(input_lines), int(max_cases))

    branch_counts = Counter()
    signal_error_sums = defaultdict(lambda: [0.0, 0.0, 0])
    per_case_report = []
    missing_debug_cases = []

    for case_index in failed_indices:
        if case_index > case_count:
            missing_debug_cases.append(case_index)
            continue

        input_line = input_lines[case_index - 1]
        debug_line = debug_lines[case_index - 1]
        a, b, c, d = parse_input_line(input_line)
        delta_hw, delta_0_hw, delta_1_hw, c_hw = parse_debug_line(debug_line)

        golden = diagnose_cubic(a, b, c, d, sqrt_iterations=sqrt_iterations, cbrt_iterations=cbrt_iterations)
        golden_delta = complex(golden["delta"])
        golden_delta_0 = complex(golden["delta_0"])
        golden_delta_1 = complex(golden["delta_1"])
        golden_c = complex(golden["C_val"])

        branch, _, _, _ = _cubic_branch(a, b, c, d)
        branch_counts[branch] += 1

        deltas = {
            "delta": abs(delta_hw - golden_delta),
            "delta_0": abs(delta_0_hw - golden_delta_0),
            "delta_1": abs(delta_1_hw - golden_delta_1),
            "C": abs(c_hw - golden_c),
        }
        abs_mean = sum(deltas.values()) / len(deltas)
        rel_mean = abs_mean / max(
            abs(golden_delta),
            abs(golden_delta_0),
            abs(golden_delta_1),
            abs(golden_c),
            1e-12,
        )

        signal_error_sums[branch][0] += abs_mean
        signal_error_sums[branch][1] += rel_mean
        signal_error_sums[branch][2] += 1

        if abs_mean > error_threshold:
            per_case_report.append((case_index, branch, abs_mean, rel_mean, deltas, (a, b, c, d), debug_line))

    print(f"Total failed cases from checker: {len(failed_indices)}")
    print(f"Debug cases available: {case_count}")
    print(f"Failed cases missing debug line: {len(missing_debug_cases)}")
    if missing_debug_cases:
        print(f"Missing indices: {missing_debug_cases}")

    print("\n--- Mean internal-signal error by branch ---")
    branch_rows = []
    for branch, count in branch_counts.most_common():
        abs_sum, rel_sum, branch_count = signal_error_sums[branch]
        if branch_count == 0:
            continue
        branch_rows.append([
            branch,
            branch_count,
            f"{abs_sum / branch_count:.6e}",
            f"{rel_sum / branch_count:.6e}",
        ])
    if branch_rows:
        _print_table(["branch", "count", "mean_abs", "mean_rel"], branch_rows)

    if per_case_report:
        print("\n--- Failed cases compared against debug ---")
        report_rows = []
        for case_index, branch, abs_mean, rel_mean, deltas, coeffs, debug_line in sorted(per_case_report, key=lambda item: item[2], reverse=True)[:10]:
            a, b, c, d = coeffs
            report_rows.append([
                case_index,
                branch,
                f"{abs_mean:.6e}",
                f"{rel_mean:.6e}",
                f"{deltas['delta']:.6e}",
                f"{deltas['delta_0']:.6e}",
                f"{deltas['delta_1']:.6e}",
                f"{deltas['C']:.6e}",
            ])
        _print_table(["case", "branch", "mean_abs", "mean_rel", "delta", "delta0", "delta1", "C"], report_rows)

    return {
        "failed_cases": len(failed_indices),
        "available_debug_cases": case_count,
        "missing_debug_cases": missing_debug_cases,
        "by_branch": dict(branch_counts),
    }


internal_summary = analyze_failed_tests_with_debug()
print("\nInternal summary:")
print(internal_summary)

Total failed cases from checker: 7
Debug cases available: 900
Failed cases missing debug line: 0

--- Mean internal-signal error by branch ---
branch                | count | mean_abs     | mean_rel    
----------------------+-------+--------------+-------------
positive-discriminant | 7     | 4.319158e+10 | 1.193828e-03

--- Failed cases compared against debug ---
case | branch                | mean_abs     | mean_rel     | delta        | delta0       | delta1       | C           
-----+-----------------------+--------------+--------------+--------------+--------------+--------------+-------------
94   | positive-discriminant | 3.023411e+11 | 8.111412e-03 | 1.209364e+12 | 1.509352e-02 | 1.535242e+02 | 1.114464e-02
268  | positive-discriminant | 1.475304e-03 | 1.265151e-04 | 1.666197e-06 | 6.014717e-10 | 2.640048e-07 | 5.899284e-03

Internal summary:
{'failed_cases': 7, 'available_debug_cases': 900, 'missing_debug_cases': [], 'by_branch': {'positive-discriminant': 7}}
